# Field Engine в Google Colab

Сжимает MoE-модель (OLMoE-1B-7B) в «полевой» артефакт: эксперты не хранятся,
собираются на лету из low-rank поля по роутеру. Подробнее — `wiki/Home.md`.

**Как запустить**
1. `Среда выполнения → Сменить среду выполнения → T4 GPU` (бесплатного тарифа хватает; GPU включится автоматически: пайплайн сам перейдёт на fp16 на T4 и включит GPU-стриминг).
2. Один раз загрузите `expert-press-update.zip` в корень Google Drive.
3. Выполните ячейки по порядку (Shift+Enter).

**Что происходит с докачкой:** все загрузки оседают на Google Drive
(`HF_HOME` → Drive) — модель скачивается **один раз**, дальше каждая сессия
только копирует файл с Drive на локальный диск (~1–2 мин) и восстанавливает
кеш калибровки, после чего стадии 1–4 пайплайна пропускаются целиком.
Код пайплайна — обычный питон, никакой адаптации: ноутбук просто его запускает.

In [ ]:
import glob, os, shutil, zipfile
from google.colab import drive

if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")

PROJ = "/content/expert-press-update"
if not os.path.isfile(f"{PROJ}/hf_pipeline.py"):
    zips = ["/content/expert-press-update.zip",
            *glob.glob("/content/drive/MyDrive/**/expert-press-update.zip",
                       recursive=True)]
    assert zips and os.path.isfile(zips[0]), (
        "загрузите expert-press-update.zip в Google Drive (или в /content) "
        "и перезапустите ячейку")
    print("unzipping:", zips[0])
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall("/content")
    assert os.path.isfile(f"{PROJ}/hf_pipeline.py"), "hf_pipeline.py не найден"
os.chdir(PROJ)

gpu = os.popen("nvidia-smi --query-gpu=name,memory.total "
               "--format=csv,noheader 2>/dev/null").read().strip()
print("project:", PROJ)
print("GPU:", gpu if gpu else
      "НЕ НАЙДЕН -> Runtime > Change runtime type > T4 GPU, затем Restart")

## 1. Пути и кеши

`HF_HOME` на Drive — это и есть главный выключатель повторных докачек: всё,
что когда-либо скачал Hugging Face, остаётся там между сессиями. Рабочая
папка пайплайна (`MOE_OUT_DIR`) — на локальном диске: быстрые случайные
чтения, а ценное (пул калибровки, артефакт) ячейка 5 синхронизирует на Drive.

In [ ]:
import os, pathlib

# ============================ НАСТРОЙКИ ==================================
REPO       = "mradermacher/OLMoE-1B-7B-0924-GGUF"  # GGUF-репозиторий
QUANT      = "Q4_K_M"       # какой квант качать (Q4_K_M | Q4_K_S | Q8_0 ...)
GGUF_FILE  = None           # точное имя файла, напр. "OLMoE-1B-7B-0924.Q4_K_M.gguf"
RANK       = 32             # ранг поля (16/32/64)
FIT_PRESET = None           # None | "fast" | "balanced" | "quality"
# =========================================================================

DRIVE_MOE  = "/content/drive/MyDrive/moe_router"  # постоянное хранилище
HF_CACHE   = f"{DRIVE_MOE}/hf_cache"              # кеш HF hub - живет между сессиями
RUN_DIR    = "/content/run"                       # рабочая папка (быстрый локальный диск)
GGUF_LOCAL = "/content/gguf_cache"                # локальная копия GGUF для быстрых чтений

os.environ["HF_HOME"]     = HF_CACHE
os.environ["MOE_OUT_DIR"] = RUN_DIR
for d in (HF_CACHE, RUN_DIR, GGUF_LOCAL, f"{DRIVE_MOE}/gguf",
          f"{DRIVE_MOE}/pool_cache", f"{DRIVE_MOE}/artifacts"):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
print("HF_HOME     ->", HF_CACHE)
print("MOE_OUT_DIR ->", RUN_DIR)

## 2. Модель: скачивается один раз, потом только с Drive

Первый запуск кладёт GGUF в кеш на Drive (~4.4 ГБ, единственная большая
докачка). Каждая следующая сессия: файл найден в кеше → копируется на
локальный диск (быстрые чтения при стриминге) → сеть не трогается вовсе.

In [ ]:
import glob, os, shutil
from huggingface_hub import HfApi, hf_hub_download

def drive_ggufs():
    return sorted(glob.glob(f"{DRIVE_MOE}/gguf/*.gguf")
                  + glob.glob(f"{HF_CACHE}/hub/models--*/snapshots/*/*.gguf"))

files = drive_ggufs()
if not files:
    print("первый запуск: скачиваю GGUF один раз -> кеш на Drive ...")
    fl = [f for f in HfApi().list_repo_files(REPO) if f.endswith(".gguf")]
    name = GGUF_FILE or next(
        (f for f in sorted(fl) if f.lower().endswith(f".{QUANT.lower()}.gguf")), None)
    assert name, f"в {REPO} нет .{QUANT}.gguf; доступно: {sorted(fl)}"
    print("file:", name)
    hf_hub_download(REPO, name)          # -> HF_HOME (Drive)
    files = drive_ggufs()
    assert files, "скачивание завершилось, но кеш пуст - проверьте HF_HOME"

def wanted(path):
    if GGUF_FILE:
        return os.path.basename(path) == GGUF_FILE
    return path.lower().endswith(f".{QUANT.lower()}.gguf")

src = next((f for f in files if wanted(f)), None)
assert src, f"подходящий GGUF не найден среди {[os.path.basename(f) for f in files]}"
GGUF_PATH = f"{GGUF_LOCAL}/{os.path.basename(src)}"
if not os.path.isfile(GGUF_PATH):
    print(f"Drive -> локальный диск (быстрые чтения): {os.path.basename(src)}")
    shutil.copyfile(src, GGUF_PATH)
print("GGUF готов:", GGUF_PATH, f"({os.path.getsize(GGUF_PATH)/1e9:.2f} GB)")

## 3. Кеш калибровки: с Drive → локальный диск

Пул пар (стадии 3–4, самый медленный стриминг) сохраняется на Drive. Если он
уже собран — эта ячейка восстанавливает его локально, и пайплайн пропускает
стадии 1–4 целиком (сам напишет `no streaming pass needed`).

In [ ]:
import os, re, shutil

TAG = re.sub(r"[^A-Za-z0-9_.-]", "_", os.path.basename(GGUF_PATH))[:60]
POOL      = f"{RUN_DIR}/cache_{TAG}"
POOL_DRVE = f"{DRIVE_MOE}/pool_cache/cache_{TAG}"

if os.path.isdir(POOL):
    print("пул уже на локальном диске:", POOL)
elif os.path.isdir(POOL_DRVE):
    print("восстанавливаю пул калибровки с Drive (стадии 3-4 будут пропущены)...")
    shutil.copytree(POOL_DRVE, POOL)
else:
    print("пула ещё нет - первый запуск соберёт его (медленный стриминг); "
          "ячейка 5 закинет его на Drive для следующих сессий")

## 4. Запуск сжатия

Обычный запуск пайплайна — без спец-адаптаций. `--device auto` включает GPU
(на T4 профиль сам переключит dtype в fp16, включит io-cache ram и
io-threads 4 — строка `hardware:` в начале лога покажет, что выбрано).
Повторный запуск с готовым пулом пробегает только fit/save/verify.

In [ ]:
CMD = f"python hf_pipeline.py --gguf {GGUF_PATH} --rank {RANK} --device auto"
if FIT_PRESET:
    CMD += f" --fit-preset {FIT_PRESET}"
print(CMD, flush=True)

!{CMD}

## 5. Синхронизация: пул и артефакт → Drive

Артефакт (`field_*_r*` — компактная модель) и пул калибровки сохраняются на
Drive: после перезапуска среды ничего пересобирать не нужно.

In [ ]:
import os, shutil

if os.path.isdir(POOL):
    print("пул калибровки -> Drive ...")
    shutil.copytree(POOL, POOL_DRVE, dirs_exist_ok=True)

ART = f"{RUN_DIR}/field_{TAG}_r{RANK}"
if os.path.isdir(ART):
    dst = f"{DRIVE_MOE}/artifacts/{os.path.basename(ART)}"
    if not os.path.isdir(dst):
        print("артефакт -> Drive:", dst)
        shutil.copytree(ART, dst)
print("готово. следующая сессия переиспользует: модель (HF-кеш), пул, артефакт")

## Заметки

- **Память.** Если словили OOM на T4-классической машине — добавьте к команде
  `--io-cache disk` (авто-выбор ram будет понижен автоматически, но явный
  флаг полезен как страховка).
- **Поболтать с артефактом:** `!python hf_chat.py --model {ART} --device auto`.
- **Новый ранг из готового пула:** поменяйте `RANK` в ячейке 1 и повторите
  шаги 4–5 — пул переиспользуется, заново стримить модель не нужно.
- Что качается и когда: `wiki/Colab.md`.